# NC Sweep — Remaining Runs

**What's done:**
- depth5 baseline: T_NC=310, feat_norm=1.063 ✓
- depth7 (3 seeds): T_NC=330-350, feat_norm=1.07-1.15 ✓  CV=2.9%
- depth2/3 (3 seeds each): NC1 close but not yet < 0.01
- GELU seed0: NC1 not collapsing (slope positive)

**This notebook runs:**
1. GELU seeds 1 and 2 (activation sweep)
2. Tanh seeds 0, 1, 2 (activation sweep)
3. Depth 2 and 3 — extended to 800 epochs, relaxed threshold NC1<0.05

**Est. runtime: ~90 min on T4**

**Fully self-contained.**

In [1]:
import torch, torchvision, time
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = '/kaggle/working/'
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
print(f'Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0)}')


Device: cuda | GPU: Tesla T4


In [2]:
transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,),(0.3081,))])
trainset  = torchvision.datasets.MNIST('/kaggle/working/data',
    train=True,  download=True, transform=transform)
testset   = torchvision.datasets.MNIST('/kaggle/working/data',
    train=False, download=True, transform=transform)
train_loader = DataLoader(trainset, batch_size=256, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=512, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f'MNIST ready: {len(trainset):,} train')


100%|██████████| 9.91M/9.91M [00:00<00:00, 39.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.09MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.8MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 14.0MB/s]

MNIST ready: 60,000 train


In [3]:
class MLP5(nn.Module):
    def __init__(self, depth=5, width=512, act_cls=nn.ReLU, num_classes=10):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(784, width), act_cls()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), act_cls()]
        self.body   = nn.Sequential(*layers)
        self.head   = nn.Linear(width, num_classes)
        self._feats = None
        self.body.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.detach()))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.head(self.body(x))
    def get_features(self, x):
        self(x); return self._feats
    def get_classifier_weights(self):
        return self.head.weight.detach()

print('MLP defined.')


MLP defined.


In [4]:
@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval()
    fl, ll = [], []
    for x, y in loader:
        fl.append(model.get_features(x.to(DEVICE)).cpu())
        ll.append(y)
    H = torch.cat(fl).float(); Y = torch.cat(ll)
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    Sw   = sum((H[Y==c]-mu_c[c]).T@(H[Y==c]-mu_c[c]) for c in range(K))/len(H)
    Sb   = M.T @ M / K
    nc1  = (torch.trace(Sw)/torch.trace(Sb).clamp(1e-10)).item()
    Mn   = F.normalize(M, dim=1)
    cos  = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool)
    nc2  = (cos[mask]-(-1.0/(K-1))).abs().mean().item()
    Wn   = F.normalize(model.get_classifier_weights().cpu(), dim=1)
    nc3  = (1-(Mn*Wn).sum(1).mean()).item()
    return {'nc1':nc1,'nc2':nc2,'nc3':nc3,
            'feat_norm':H.norm(dim=1).mean().item()}

def evaluate(model, loader):
    model.eval(); correct=total=0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(DEVICE),y.to(DEVICE)
            correct += (model(x).argmax(1)==y).sum().item()
            total   += len(y)
    return correct/total

def run_twophase(model, name, lr=1e-3, wd=1e-4,
                 phase1=200, phase2=400, nc_every=10, nc_threshold=0.01):
    model = model.to(DEVICE)
    K = 10; rows = []; terminal = False; t0 = time.time()
    for phase, loss_fn, n_ep in [
        (1, 'ce',  phase1),
        (2, 'mse', phase2),
    ]:
        opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep)
        offset = phase1 if phase == 2 else 0
        for ep_l in range(1, n_ep+1):
            ep = offset + ep_l
            model.train()
            for x, y in train_loader:
                x,y = x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                loss = F.mse_loss(logits,F.one_hot(y,K).float()) \
                       if loss_fn=='mse' else F.cross_entropy(logits,y)
                loss.backward(); opt.step()
            sched.step()
            if ep_l % nc_every == 0 or ep_l == n_ep:
                tr = evaluate(model, train_loader)
                te = evaluate(model, test_loader)
                if tr >= 0.99 and not terminal:
                    terminal = True
                    print(f'  [{name}] Terminal ep={ep}')
                nc = compute_nc(model, train_loader) if terminal else \
                     {'nc1':None,'nc2':None,'nc3':None,'feat_norm':None}
                rows.append({'epoch':ep,'phase':phase,'train':tr,'test':te,**nc})
                nc1s = f"{nc['nc1']:.5f}" if nc['nc1'] else 'N/A'
                fns  = f"{nc['feat_norm']:.3f}" if nc['feat_norm'] else 'N/A'
                print(f'  ep={ep:>4} tr={tr:.4f} nc1={nc1s} fn={fns} '
                      f't={(time.time()-t0)/60:.1f}m')
                if nc['nc1'] is not None and nc['nc1'] < nc_threshold:
                    fn_val = nc['feat_norm']
                    print(f'  *** T_NC={ep} fn={fn_val:.4f} (NC1<{nc_threshold})')
                    return pd.DataFrame(rows), ep, fn_val
    return pd.DataFrame(rows), None, None

print('run_twophase ready. nc_threshold parameter added.')


run_twophase ready. nc_threshold parameter added.


In [5]:
results = []

# ── GELU seeds 1 and 2 (seed 0 ran but GELU nc1 was increasing) ──────────
print('=== GELU activation seeds 1,2 ===')
for seed in [1, 2]:
    name = f'GELU-s{seed}'
    print(f'\n--- {name} ---')
    torch.manual_seed(seed)
    model = MLP5(depth=5, width=512, act_cls=nn.GELU)
    df, t_nc, fn = run_twophase(model, name, lr=1e-3, wd=1e-4,
                                phase1=200, phase2=400)
    df.to_csv(f'{SAVE_DIR}actGELU_s{seed}.csv', index=False)
    results.append({'act':'GELU','seed':seed,'T_NC':t_nc,'feat_norm':fn,
                    'test_acc':df.test.iloc[-1]})
    print(f'  => T_NC={t_nc}  feat_norm={fn}')

# ── Tanh seeds 0,1,2 ─────────────────────────────────────────────────────
print('\n=== Tanh activation seeds 0,1,2 ===')
for seed in [0, 1, 2]:
    name = f'Tanh-s{seed}'
    print(f'\n--- {name} ---')
    torch.manual_seed(seed)
    model = MLP5(depth=5, width=512, act_cls=nn.Tanh)
    df, t_nc, fn = run_twophase(model, name, lr=1e-3, wd=1e-4,
                                phase1=200, phase2=400)
    df.to_csv(f'{SAVE_DIR}actTanh_s{seed}.csv', index=False)
    results.append({'act':'Tanh','seed':seed,'T_NC':t_nc,'feat_norm':fn,
                    'test_acc':df.test.iloc[-1]})
    print(f'  => T_NC={t_nc}  feat_norm={fn}')

# ── Depth 2 and 3 at relaxed threshold NC1<0.05 ──────────────────────────
# Previous run: depth2/3 reached NC1~0.01-0.04 but not <0.01
# NC1<0.05 is still clear partial collapse — valid data point
print('\n=== Depth 2,3 — relaxed threshold NC1<0.05 ===')
for depth in [2, 3]:
    for seed in range(3):
        name = f'depth{depth}-s{seed}-extended'
        print(f'\n--- depth={depth} seed={seed} ---')
        torch.manual_seed(seed)
        model = MLP5(depth=depth, width=512, act_cls=nn.ReLU)
        # Longer phase2 (600 ep) and relaxed threshold
        df, t_nc, fn = run_twophase(model, name, lr=1e-3, wd=1e-4,
                                    phase1=200, phase2=600,
                                    nc_threshold=0.05)
        df.to_csv(f'{SAVE_DIR}depth{depth}_s{seed}_ext.csv', index=False)
        results.append({'depth':depth,'seed':seed,'T_NC':t_nc,'feat_norm':fn,
                        'test_acc':df.test.iloc[-1]})
        print(f'  => T_NC={t_nc}  feat_norm={fn}')

pd.DataFrame(results).to_csv(SAVE_DIR+'sweep_remaining.csv', index=False)
print('\nAll runs complete. Saved: sweep_remaining.csv')

# Summary
print('\n=== THRESHOLD SUMMARY ===')
all_fns = [1.063, 1.117, 1.149, 1.069]  # from previous runs
for r in results:
    if r.get('feat_norm'):
        all_fns.append(r['feat_norm'])
        print(f"  {r}: fn={r['feat_norm']:.4f}")
if len(all_fns) > 1:
    cv = np.std(all_fns)/np.mean(all_fns)
    print(f'\nAll feat_norm at T_NC: mean={np.mean(all_fns):.4f} '
          f'std={np.std(all_fns):.4f} CV={cv:.3f}')
    print('CONSISTENT THRESHOLD!' if cv < 0.15 else f'CV={cv:.3f} — varies')


=== GELU activation seeds 1,2 ===

--- GELU-s1 ---
  [GELU-s1] Terminal ep=10
  ep=  10 tr=0.9946 nc1=0.21809 fn=20.109 t=1.3m
  ep=  20 tr=0.9924 nc1=0.12469 fn=20.575 t=2.6m
  ep=  30 tr=0.9957 nc1=0.09798 fn=18.361 t=3.9m
  ep=  40 tr=0.9970 nc1=0.08611 fn=20.737 t=5.2m
  ep=  50 tr=0.9958 nc1=0.08659 fn=18.650 t=6.5m
  ep=  60 tr=0.9975 nc1=0.08312 fn=17.910 t=7.8m
  ep=  70 tr=0.9961 nc1=0.08298 fn=18.004 t=9.1m
  ep=  80 tr=0.9995 nc1=0.07337 fn=17.887 t=10.4m
  ep=  90 tr=1.0000 nc1=0.06541 fn=18.262 t=11.6m
  ep= 100 tr=0.9993 nc1=0.07427 fn=16.590 t=12.9m
  ep= 110 tr=0.9999 nc1=0.06732 fn=17.106 t=14.2m
  ep= 120 tr=0.9994 nc1=0.07923 fn=15.592 t=15.5m
  ep= 130 tr=1.0000 nc1=0.07771 fn=15.982 t=16.8m
  ep= 140 tr=1.0000 nc1=0.07851 fn=15.887 t=18.1m
  ep= 150 tr=1.0000 nc1=0.08178 fn=15.846 t=19.3m
  ep= 160 tr=1.0000 nc1=0.08711 fn=15.243 t=20.6m
  ep= 170 tr=1.0000 nc1=0.08878 fn=15.069 t=21.9m
  ep= 180 tr=1.0000 nc1=0.09135 fn=14.858 t=23.2m
  ep= 190 tr=1.0000 nc1=0.092